In [ ]:
# Core
import os
import json
from typing import Dict

# Database
import sqlite3

# Object storage (MinIO)
from minio import Minio

# LLM provider (example: OpenAI-compatible)
from openai import OpenAI


In [ ]:
SQLITE_DB_PATH = "artworks.db"

def get_db_connection():
    return sqlite3.connect(SQLITE_DB_PATH)


In [ ]:
minio_client = Minio(
    endpoint="localhost:9000",
    access_key="MINIO_ACCESS_KEY",
    secret_key="MINIO_SECRET_KEY",
    secure=False
)

ARTWORK_BUCKET = "artworks"


In [ ]:
REFLECTION_SYSTEM_PROMPT = """
You are a reflective companion helping people engage with artwork.

You do not explain or teach.
You do not analyze or critique.
You do not claim artistic intent.

You offer gentle, human reflections that help viewers feel comfortable
having their own thoughts about the artwork.

Tone:
- Calm
- Non-authoritative
- Open-ended
- Emotionally accessible

Constraints:
- No art history unless explicitly provided
- No jargon
- No instructions
- One short paragraph only
"""


In [ ]:
REFLECTION_USER_PROMPT = """
Artwork Title: "{title}"
Artist: "{artist}"

Artwork Description:
{description}

Write a short reflection that feels like a quiet thought someone might have
while standing in front of this artwork.
"""


In [ ]:
def build_prompt(title: str, artist: str, description: str) -> Dict[str, str]:
    return {
        "system": REFLECTION_SYSTEM_PROMPT.strip(),
        "user": REFLECTION_USER_PROMPT.format(
            title=title.strip(),
            artist=artist.strip() if artist else "Unknown",
            description=description.strip()
        )
    }


In [ ]:
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)


In [ ]:
def generate_reflection(prompt: Dict[str, str]) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": prompt["system"]},
            {"role": "user", "content": prompt["user"]}
        ],
        temperature=0.7,
        max_tokens=120
    )

    return response.choices[0].message.content.strip()


In [ ]:
test_prompt = build_prompt(
    title="Untitled (Blue and Gray)",
    artist="Mark Rothko",
    description=(
        "A large canvas dominated by soft, layered fields of blue and gray, "
        "with blurred edges that create a sense of depth and stillness."
    )
)

reflection = generate_reflection(test_prompt)
print(reflection)


In [ ]:
def save_reflection(artwork_id: int, reflection: str):
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute(
        "UPDATE artworks SET reflection_text = ? WHERE id = ?",
        (reflection, artwork_id)
    )

    conn.commit()
    conn.close()
